In [1]:
import pandas as pd
from config import PATH_TO_EMBEDDING
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np

In [2]:
df = pd.read_csv("dataset/legal_truncated_corpus.csv", encoding="utf-16",engine="python")

df.drop(labels=['Unnamed: 0'], axis="columns", inplace=True)
laws = df["context"].to_list()

In [6]:
embedder = SentenceTransformer(PATH_TO_EMBEDDING)
embeddings = embedder.encode(laws[:100000],show_progress_bar=True)

Batches:   0%|          | 0/3125 [00:00<?, ?it/s]

In [7]:
m = 8
nbits = 8
d = embeddings.shape[1]
nlist = 64
quantizer = faiss.IndexFlatL2(d)
index = faiss.IndexIVFPQ(quantizer, d, nlist, m, nbits)
index.train(np.array(embeddings))
index.add(np.array(embeddings))

In [8]:
faiss.write_index(index, "laws_first_100k.index")

In [14]:
laws = laws[:100000]
import json
#lưu list laws thành file json:
#Định dạng 
with open("laws_first_100k.json", 'w', encoding='utf-8') as f:
    json.dump(laws, f, ensure_ascii=False)

